In [1]:
import numpy as np
import pandas as pd
import pydicom
%matplotlib inline
import matplotlib.pyplot as plt
import keras 

from keras.models import model_from_json
from skimage.transform import resize
from keras.applications.vgg16 import preprocess_input

Using TensorFlow backend.


In [4]:
# This function reads in a .dcm file, checks the important fields for our device, and returns a numpy array
# of just the imaging data
def check_dicom(filename): 
    ds = pydicom.dcmread(filename)     
    
    if ds.PatientPosition not in ['AP', 'PA']:
        msg = 'Invalid Image. Reason: Patient Position is {}'.format(ds.PatientPosition)
        print(msg)
        return None, msg
    

    if ds.Modality != 'DX':
        msg = 'Invalid Image. Reason: Modality is {}'.format(ds.Modality)
        print(msg)
        return None, msg
    

    if ds.BodyPartExamined not in ['CHEST', 'RIBCAGE']:
        msg = 'Invalid Image. Reason: body part is {}'.format(ds.BodyPartExamined)
        print(msg)
        return None, msg
    
    img = ds.pixel_array
    return img, ds.StudyDescription
    
    
# This function takes the numpy array output by check_dicom and 
# runs the appropriate pre-processing needed for our model input
def preprocess_image(img,img_mean,img_std,img_size): 
    rescale = 1. / 255.0
    
    proc_img = img * rescale
    proc_img = resize(proc_img, (img_size[1], img_size[2]), anti_aliasing=False)
    proc_img = proc_img.reshape((img_size[0], img_size[1], img_size[2], 1))

    # Reformat data to match the input data format of the model
    proc_img = np.repeat(proc_img, img_size[3], axis=3)
    
    # https://stackoverflow.com/questions/47555829/preprocess-input-method-in-keras/47556342#47556342
    proc_img = preprocess_input(proc_img)
    
    return proc_img

# This function loads in our trained model w/ weights and compiles it 
def load_model(model_path, weight_path):
    with open("my_model.json", "r") as json:
        model_json = json.read()
        
    model = model_from_json(model_json)
    model.load_weights(weight_path)
    
    return model

# This function uses our device's threshold parameters to predict whether or not
# the image shows the presence of pneumonia using our trained model
def predict_image(model, img, thresh): 
    prob = model.predict(img, verbose = True)
    pred = prob > thresh
    prediction = pred[0][0]
    
    return prediction  

In [5]:
test_dicoms = ['test1.dcm','test2.dcm','test3.dcm','test4.dcm','test5.dcm','test6.dcm']

model_path = "my_model.json"
weight_path = "{}_my_model.best.hdf5".format('xray_class') 

IMG_SIZE=(1,224,224,3) # This might be different if you did not use vgg16
img_mean = None # loads the mean image value they used during training preprocessing
img_std = None # loads the std dev image value they used during training preprocessing

my_model = load_model(model_path, weight_path)
thresh = 0.85 #loads the threshold they chose for model classification 

# use the .dcm files to test your prediction
for i in test_dicoms:
    
    img = np.array([])
    img, desc = check_dicom(i)
    
    if img is None:
        print(f"{i} File not found or incomplete")
        continue
        
    img_proc = preprocess_image(img,img_mean,img_std,IMG_SIZE)
    pred = predict_image(my_model,img_proc,thresh)
    print(f"DICOM: {i}\nModel Prediction:{pred} - Radiologist:{desc}")
    print("---")

1/1 [==============================] - 0s 446ms/step
DICOM: test1.dcm
Model Prediction:True - Radiologist:No Finding
---
1/1 [==============================] - 0s 491ms/step
DICOM: test2.dcm
Model Prediction:True - Radiologist:Cardiomegaly
---
1/1 [==============================] - 0s 493ms/step
DICOM: test3.dcm
Model Prediction:True - Radiologist:Effusion
---
1/1 [==============================] - 0s 488ms/step
DICOM: test4.dcm
Model Prediction:True - Radiologist:No Finding
---
Invalid Image. Reason: Modality is CT
test5.dcm File not found or incomplete
Invalid Image. Reason: Patient Position is XX
test6.dcm File not found or incomplete
